# 03 · Silver → Gold: star schema for the FP&A semantic model

**Dimensions:** `dim_date`, `dim_product`, `dim_customer`, `dim_market`, `dim_account`, `dim_cost_center`

**Facts:**
* `fact_sales` – invoice lines converted to PLN with the NBP rate of the previous business day
* `fact_pnl` – monthly P&L, **Actual and Budget in one table** (`scenario` column).
  Sign convention: revenue positive, costs negative → any P&L subtotal is a simple SUM,
  and `Actual − Budget` is always "positive = favourable".
* `fact_revenue_bridge` – revenue variance split into **volume, price and FX effects**
  (product × market × month, closed months only)

In [ ]:
LAST_CLOSED_MONTH = ""   # "YYYY-MM-01"; empty = last month with sales invoices
DATE_FROM, DATE_TO = "2025-01-01", "2026-12-31"

In [ ]:
from pyspark.sql import functions as F


def save_table(df, name: str) -> None:
    (df.write.mode("overwrite").format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(name))
    print(f"✔ {name}: {spark.read.table(name).count():,} rows")


sales = spark.read.table("silver_sales_invoices")
if not LAST_CLOSED_MONTH:
    LAST_CLOSED_MONTH = str(sales.agg(F.max(F.trunc("invoice_date", "month"))).first()[0])
print("Last closed month:", LAST_CLOSED_MONTH)

## Dimensions

In [ ]:
dim_date = (spark.createDataFrame([(DATE_FROM, DATE_TO)], ["d0", "d1"])
    .select(F.explode(F.sequence(F.to_date("d0"), F.to_date("d1"))).alias("date"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("month_no", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("month_start", F.trunc("date", "month"))
    .withColumn("is_closed_month", F.trunc("date", "month") <= F.to_date(F.lit(LAST_CLOSED_MONTH))))
save_table(dim_date, "dim_date")

save_table(spark.read.table("silver_products"), "dim_product")
save_table(spark.read.table("silver_customers"), "dim_customer")
save_table(spark.read.table("silver_customers").select("market", "currency").distinct(), "dim_market")
save_table(spark.read.table("silver_accounts"), "dim_account")
save_table(spark.read.table("silver_cost_centers"), "dim_cost_center")

## fact_sales – invoices in PLN

In [ ]:
customers = spark.read.table("silver_customers").select("customer_id", "market")
fx = spark.read.table("silver_fx_rates_daily").select(
    "currency", F.col("date").alias("invoice_date"), "rate_for_invoice_date", "nbp_table_for_invoice")

fact_sales = (sales
    .join(customers, "customer_id", "left")
    .join(fx, ["currency", "invoice_date"], "left")
    .withColumn("fx_rate", F.when(F.col("currency") == "PLN", F.lit(1)).otherwise(F.col("rate_for_invoice_date"))
                .cast("decimal(10,4)"))
    .withColumn("net_amount_pln", F.round(F.col("net_amount") * F.col("fx_rate"), 2).cast("decimal(18,2)"))
    .select("invoice_no", "invoice_line", "invoice_date", "doc_type", "customer_id", "market", "product_id",
            "quantity_t", "unit_price", "currency", F.col("net_amount").alias("net_amount_cur"),
            "fx_rate", "nbp_table_for_invoice", "net_amount_pln"))

missing_fx = fact_sales.filter(F.col("fx_rate").isNull()).count()
assert missing_fx == 0, f"{missing_fx} invoices without FX rate – run notebook 01 with an earlier START_DATE"
save_table(fact_sales, "fact_sales")

## fact_pnl – Actual and Budget in one table

In [ ]:
REVENUE_ACCOUNT, REVENUE_CC = "701-01", "SAL-01"
pnl_cols = ["month_start", "scenario", "account", "cost_center", "product_id", "market", "quantity_t", "amount_pln"]

actual_revenue = (spark.read.table("fact_sales")
    .groupBy(F.trunc("invoice_date", "month").alias("month_start"), "product_id", "market")
    .agg(F.sum("quantity_t").alias("quantity_t"), F.sum("net_amount_pln").alias("amount_pln"))
    .withColumn("scenario", F.lit("Actual"))
    .withColumn("account", F.lit(REVENUE_ACCOUNT))
    .withColumn("cost_center", F.lit(REVENUE_CC)))

actual_costs = (spark.read.table("silver_gl_costs")
    .groupBy(F.trunc("posting_date", "month").alias("month_start"), "account", "cost_center")
    .agg((-F.sum("amount_pln")).alias("amount_pln"))
    .withColumn("scenario", F.lit("Actual"))
    .withColumn("product_id", F.lit(None).cast("string"))
    .withColumn("market", F.lit(None).cast("string"))
    .withColumn("quantity_t", F.lit(None).cast("decimal(18,3)")))

assumptions = spark.read.table("silver_budget_assumptions").select("year", "budget_eur_pln")
budget_sales = (spark.read.table("silver_budget_sales")
    .join(assumptions, "year")
    .withColumn("budget_fx", F.when(F.col("currency") == "PLN", F.lit(1.0)).otherwise(F.col("budget_eur_pln"))))

budget_revenue = (budget_sales
    .withColumn("amount_pln", F.col("quantity_t") * F.col("price_per_t") * F.col("budget_fx"))
    .withColumn("scenario", F.lit("Budget"))
    .withColumn("account", F.lit(REVENUE_ACCOUNT))
    .withColumn("cost_center", F.lit(REVENUE_CC)))

budget_costs = (spark.read.table("silver_budget_costs")
    .withColumn("amount_pln", -F.col("amount_pln"))
    .withColumn("scenario", F.lit("Budget"))
    .withColumn("product_id", F.lit(None).cast("string"))
    .withColumn("market", F.lit(None).cast("string"))
    .withColumn("quantity_t", F.lit(None).cast("decimal(18,3)")))

fact_pnl = (actual_revenue.select(pnl_cols)
    .unionByName(actual_costs.select(pnl_cols))
    .unionByName(budget_revenue.select(pnl_cols))
    .unionByName(budget_costs.select(pnl_cols))
    .withColumn("quantity_t", F.col("quantity_t").cast("decimal(18,3)"))
    .withColumn("amount_pln", F.round("amount_pln", 2).cast("decimal(18,2)")))
save_table(fact_pnl, "fact_pnl")

## fact_revenue_bridge – volume / price / FX effects

For every product × market × month (Q = quantity, P = price in invoice currency, F = EUR/PLN rate):

| Effect | Formula |
|---|---|
| Volume | (Q<sub>act</sub> − Q<sub>bud</sub>) × P<sub>bud</sub> × F<sub>bud</sub> |
| Price  | (P<sub>act</sub> − P<sub>bud</sub>) × Q<sub>act</sub> × F<sub>bud</sub> |
| FX     | (F<sub>act</sub> − F<sub>bud</sub>) × P<sub>act</sub> × Q<sub>act</sub> |

Budget revenue + Volume + Price + FX = Actual revenue (exactly).
P<sub>act</sub> and F<sub>act</sub> are volume-weighted averages from the invoices.

In [ ]:
act = (spark.read.table("fact_sales")
    .groupBy(F.trunc("invoice_date", "month").alias("month_start"), "product_id", "market")
    .agg(F.sum("quantity_t").cast("double").alias("q_act"),
         F.sum("net_amount_cur").cast("double").alias("rev_act_cur"),
         F.sum("net_amount_pln").cast("double").alias("rev_act_pln")))

bud = budget_sales.select(
    "month_start", "product_id", "market",
    F.col("quantity_t").cast("double").alias("q_bud"),
    F.col("price_per_t").cast("double").alias("p_bud"),
    F.col("budget_fx").cast("double").alias("f_bud"))

keys = ["month_start", "product_id", "market"]
b = (bud.join(act, keys, "full")
    .filter(F.col("month_start") <= F.to_date(F.lit(LAST_CLOSED_MONTH)))
    .fillna(0.0, ["q_act", "rev_act_cur", "rev_act_pln", "q_bud"])
    .withColumn("p_act_raw", F.when(F.col("q_act") != 0, F.col("rev_act_cur") / F.col("q_act")))
    .withColumn("p_bud", F.coalesce("p_bud", "p_act_raw"))
    .withColumn("p_act", F.coalesce("p_act_raw", "p_bud"))
    .withColumn("f_act", F.when(F.col("rev_act_cur") != 0, F.col("rev_act_pln") / F.col("rev_act_cur")))
    .withColumn("f_bud", F.coalesce("f_bud", "f_act"))
    .withColumn("f_act", F.coalesce("f_act", "f_bud")))

fact_bridge = b.select(
    *keys,
    F.col("q_bud").alias("budget_quantity_t"),
    F.col("q_act").alias("actual_quantity_t"),
    (F.col("q_bud") * F.col("p_bud") * F.col("f_bud")).alias("budget_revenue_pln"),
    ((F.col("q_act") - F.col("q_bud")) * F.col("p_bud") * F.col("f_bud")).alias("volume_effect_pln"),
    ((F.col("p_act") - F.col("p_bud")) * F.col("q_act") * F.col("f_bud")).alias("price_effect_pln"),
    ((F.col("f_act") - F.col("f_bud")) * F.col("p_act") * F.col("q_act")).alias("fx_effect_pln"),
    F.col("rev_act_pln").alias("actual_revenue_pln"),
)
fact_bridge = fact_bridge.select(
    *keys, "budget_quantity_t", "actual_quantity_t",
    *[F.round(c, 2).cast("decimal(18,2)").alias(c) for c in
      ["budget_revenue_pln", "volume_effect_pln", "price_effect_pln", "fx_effect_pln", "actual_revenue_pln"]])

# reconciliation: budget + effects must equal actual
check = fact_bridge.agg(F.sum(F.col("budget_revenue_pln") + F.col("volume_effect_pln") + F.col("price_effect_pln")
                              + F.col("fx_effect_pln") - F.col("actual_revenue_pln")).alias("diff")).first().diff
print(f"Bridge reconciliation difference: {check:,.2f} PLN (rounding only)")
save_table(fact_bridge, "fact_revenue_bridge")

In [ ]:
# helper dimension for the waterfall chart (Budget → Volume → Price → FX → Actual)
bridge_steps = spark.createDataFrame([(1, "Budget"), (2, "Volume"), (3, "Price"), (4, "FX")], ["step_order", "step"])
save_table(bridge_steps, "dim_bridge_step")

In [ ]:
# Quick look: yearly P&L, Actual vs Budget for closed months (million PLN)
acc = spark.read.table("dim_account").select("account", "pl_line", "pl_line_order")
summary = (spark.read.table("fact_pnl")
    .filter(F.col("month_start") <= F.to_date(F.lit(LAST_CLOSED_MONTH)))
    .join(acc, "account")
    .groupBy(F.year("month_start").alias("year"), "pl_line_order", "pl_line")
    .pivot("scenario", ["Actual", "Budget"]).agg(F.round(F.sum("amount_pln") / 1e6, 1))
    .withColumn("Variance", F.round(F.col("Actual") - F.col("Budget"), 1))
    .orderBy("year", "pl_line_order"))
display(summary)